# 01 - Verify Symile ECG-CXR pairing

The Symile paper, Section 5.3 and Appendix I.4, states that each sample contains an ECG within 24 hours of the patient's hospital admission and a CXR 24-72 hours after that admission; the earliest ECG and CXR are selected for each admission. Thus a positive row is matched through the same `subject_id` and `hadm_id`, but the measurements are temporally related rather than simultaneous.

This notebook verifies that statement against the released CSV paths and NPY row identities. The retrieval `label` is only a candidate-match indicator, not a diagnosis.

In [ ]:
from pathlib import Path
import json
import math
import os
import re

import numpy as np
import pandas as pd

ARTIFACT_ROOT = Path(os.environ.get(
    "ECG_CXR_BRIDGE_ARTIFACT_ROOT",
    "/path/to/ecg_cxr_bridge_artifacts",
))
config_path = ARTIFACT_ROOT / "config.json"
if config_path.exists():
    config = json.loads(config_path.read_text())
    print("Loaded configuration:", config_path)
else:
    # Self-bootstrap when notebook 00 has not yet been run.
    symile_root = Path(os.environ.get(
        "SYMILE_ROOT",
        "/path/to/symile-mimic/1.0.0",
    ))
    hf_cache = Path(os.environ.get(
        "HF_HOME", "/path/to/huggingface_cache"
    ))
    config = {
        "pipeline_version": "symile_ecg_cxr_bridge_v1",
        "symile_root": str(symile_root),
        "artifact_root": str(ARTIFACT_ROOT),
        "hf_cache": str(hf_cache),
        "hubert_model_id": "Edoardo-BS/hubert-ecg-base",
        "cxr_model_id": "microsoft/rad-dino",
        "splits": ["train", "val", "val_retrieval", "test"],
        "seed": 43,
    }
    for directory in [
        ARTIFACT_ROOT, ARTIFACT_ROOT / "manifests",
        ARTIFACT_ROOT / "features", ARTIFACT_ROOT / "models",
        ARTIFACT_ROOT / "code", ARTIFACT_ROOT / "reports",
        ARTIFACT_ROOT / "external", hf_cache,
    ]:
        directory.mkdir(parents=True, exist_ok=True)
    config_path.write_text(json.dumps(config, indent=2))
    print("Created missing configuration:", config_path)
SYMILE_ROOT = Path(config["symile_root"])
SPLITS = config["splits"]
if not SYMILE_ROOT.exists():
    raise FileNotFoundError(f"Symile dataset root does not exist: {SYMILE_ROOT}")
print("SYMILE_ROOT:", SYMILE_ROOT)


In [ ]:
def npy_header(path):
    with Path(path).open("rb") as stream:
        version = np.lib.format.read_magic(stream)
        reader = np.lib.format.read_array_header_1_0 if version == (1, 0) else np.lib.format.read_array_header_2_0
        shape, fortran, dtype = reader(stream)
        header_bytes = stream.tell()
    expected = header_bytes + math.prod(shape) * dtype.itemsize
    return {"shape": tuple(shape), "dtype": str(dtype),
            "expected_bytes": expected, "actual_bytes": Path(path).stat().st_size,
            "complete": expected == Path(path).stat().st_size}

rows = []
for split in SPLITS:
    for prefix in ["ecg", "cxr", "hadm_id"]:
        path = SYMILE_ROOT / "data_npy" / split / f"{prefix}_{split}.npy"
        if not path.exists():
            raise FileNotFoundError(path)
        row = {"split": split, "array": prefix, "path": str(path), **npy_header(path)}
        rows.append(row)
integrity = pd.DataFrame(rows)
display(integrity)
if not integrity["complete"].all():
    raise RuntimeError("One or more ECG/CXR bridge arrays are truncated.")


In [ ]:
def subject_from_path(value):
    matches = [part for part in Path(str(value)).parts if re.fullmatch(r"p\d{8}", part)]
    if len(matches) != 1:
        raise ValueError(f"Cannot identify one subject in path: {value}")
    return int(matches[0][1:])

csv_frames = {}
pair_checks = {}
for split in SPLITS:
    frame = pd.read_csv(SYMILE_ROOT / f"{split}.csv", low_memory=False)
    required = {"subject_id", "hadm_id", "ecg_path", "cxr_path"}
    if not required.issubset(frame.columns):
        raise ValueError(f"{split} is missing {sorted(required - set(frame.columns))}")
    ecg_subject = frame["ecg_path"].map(subject_from_path).to_numpy(np.int64)
    cxr_subject = frame["cxr_path"].map(subject_from_path).to_numpy(np.int64)
    row_subject = frame["subject_id"].to_numpy(np.int64)
    assert np.array_equal(ecg_subject, row_subject), f"{split}: ECG subject mismatch"
    assert np.array_equal(cxr_subject, row_subject), f"{split}: CXR subject mismatch"
    ids = np.load(SYMILE_ROOT / "data_npy" / split / f"hadm_id_{split}.npy", mmap_mode="r")
    assert np.array_equal(frame["hadm_id"].to_numpy(np.int64), np.asarray(ids))
    csv_frames[split] = frame
    pair_checks[split] = {"rows": len(frame), "unique_subjects": int(frame.subject_id.nunique()),
                          "unique_admissions": int(frame.hadm_id.nunique()),
                          "path_subject_match": True, "npy_row_match": True}
print(json.dumps(pair_checks, indent=2))


In [ ]:
for split in ["val_retrieval", "test"]:
    frame = csv_frames[split]
    groups = frame.groupby("label_hadm_id", sort=False)
    assert groups.size().eq(10).all(), f"{split}: not ten candidates per query"
    assert groups["label"].sum().eq(1).all(), f"{split}: not one positive per query"
    positives = frame[frame["label"] == 1]
    assert (positives["hadm_id"] == positives["label_hadm_id"]).all()

patient_sets = {name: set(csv_frames[name]["subject_id"]) for name in ["train", "val", "test"]}
for left, right in [("train", "val"), ("train", "test"), ("val", "test")]:
    assert not patient_sets[left] & patient_sets[right], f"Patient leakage: {left}/{right}"
print("Candidate labels and patient-disjoint splits verified.")


In [ ]:
report = {
    "paper_interpretation": ("ECG and CXR are selected from the same patient admission; "
        "ECG is within +/-24h and CXR is 24-72h after admission."),
    "pair_checks": pair_checks,
    "patient_disjoint": True,
    "retrieval_label_meaning": "one true corresponding encounter among ten candidates",
}
path = ARTIFACT_ROOT / "reports" / "01_pair_verification.json"
path.write_text(json.dumps(report, indent=2))
print("Saved:", path)
